## Manual validation - Exploration of results

In [41]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent.resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [42]:
from src.paths import VALIDATION_SAMPLE

df = pd.read_excel(VALIDATION_SAMPLE)

In [ ]:
# Total unique topics and subtopics
print(len(df['topic'].unique()))
print(len(df['subtopics'].unique()))

21
87


In [ ]:
def generate_invalid_topics_table(classified_path: str = VALIDATION_SAMPLE) -> pd.DataFrame:
    """
    Generate and print a table showing topics with invalid topic/subtopic counts.
    
    Columns:
    - Tópico: Topic name
    - Inválidos: Count of posts with invalid topic OR invalid subtopic
    - Total: Total count of posts for this topic
    - Diferença: Total - Inválidos (valid posts)
    
    Args:
        classified_path: Path to CLASSIFIED_POSTS CSV/Excel file
    
    Returns:
        DataFrame with the table
    """
    # Read data
    df = pd.read_excel(classified_path)
    
    # Create a column to identify invalid posts
    df['is_invalid'] = (df['topic_veredict'] != True) | (df['subtopic_veredict'] != True)
    
    # Group by topic
    results = []
    
    for topic in df['topic'].unique():
        topic_df = df[df['topic'] == topic]
        
        total = len(topic_df)
        invalidos = topic_df['is_invalid'].sum()
        diferenca = total - invalidos
        
        results.append({
            'Tópico': topic,
            'Inválidos': int(invalidos),
            'Total': total,
            'Diferença': diferenca
        })
    
    # Create DataFrame
    result_df = pd.DataFrame(results)
    
    # Sort by Inválidos descending
    result_df = result_df.sort_values('Inválidos', ascending=False).reset_index(drop=True)
    
    # Add totals row
    totals = {
        'Tópico': 'Total',
        'Inválidos': result_df['Inválidos'].sum(),
        'Total': result_df['Total'].sum(),
        'Diferença': result_df['Diferença'].sum()
    }
    result_df = pd.concat([result_df, pd.DataFrame([totals])], ignore_index=True)
    
    # Print formatted table
    print("=" * 100)
    print("Invalid topics table")
    print("=" * 100)
    print(result_df.to_string(index=False))
    print("=" * 100)
    
    return result_df


df_invalid = generate_invalid_topics_table()

Invalid topics table
                                                          Tópico  Inválidos  Total  Diferença
                   OpenSSL installation/build and runtime errors         11     55         44
                       Block cipher modes, IV/nonce, and padding          8     21         13
                       Crypto library/API implementation in code          6     23         17
                       Application data encryption & key storage          4     12          8
                     File handling, integrity checks & checksums          3     15         12
      Public-key cryptography: encryption and digital signatures          2     18         16
       Client-server secure communication (HTTP/HTTPS, sessions)          2     14         12
    Key protection on devices (memory/disk access, threat model)          2     11          9
             Password hashing with salt (bcrypt, rainbow tables)          2     25         23
                   X.509 certificates, 

In [44]:
df.groupby('topic')['topic_veredict'].mean().sort_values(ascending=False)

topic
Application data encryption & key storage                           1.000000
Conceptual crypto explanations and learning                         1.000000
Encoding and byte/string conversions (Base64, binary)               1.000000
Cryptosystem design pitfalls & attack resistance                    1.000000
Cryptographic hash functions & collisions                           1.000000
X.509 certificates, signing, and trust chains                       1.000000
Security vs performance (cost, speed, hardware limits)              1.000000
Secure random number generation (entropy, seeding, PRNG)            1.000000
Key sizes and security parameters selection                         1.000000
Number theory for crypto (primes, factoring, modular arithmetic)    1.000000
User authentication and account management                          1.000000
Password hashing with salt (bcrypt, rainbow tables)                 1.000000
Public-key cryptography: encryption and digital signatures          0.

In [45]:
df['topic_veredict'].mean()

np.float64(0.9323943661971831)

In [46]:
df['subtopic_veredict'].mean()

np.float64(0.8760563380281691)

In [47]:
def calcular_kappa_segmento(df_segmento, nome_dupla):
    n = len(df_segmento)
    pe = 0.50  

    concordancias_topico = (
        df_segmento['topic_validation_1'] == df_segmento['topic_validation_2']
    ).sum()
    po_topico = concordancias_topico / n
    kappa_topico = (po_topico - pe) / (1 - pe)

    concordancias_subtopico = (
        df_segmento['subtopic_validation_1'] == df_segmento['subtopic_validation_2']
    ).sum()
    po_subtopico = concordancias_subtopico / n
    kappa_subtopico = (po_subtopico - pe) / (1 - pe)

    print(f"\nResultados — {nome_dupla}")
    print(f"Total de registros: {n}")
    print("-" * 40)
    print("TÓPICOS:")
    print(f"  - Porcentagem de Concordância: {po_topico:.2%}")
    print(f"  - Valor de Kappa: {kappa_topico:.3f}")
    print("\nSUBTÓPICOS:")
    print(f"  - Porcentagem de Concordância: {po_subtopico:.2%}")
    print(f"  - Valor de Kappa: {kappa_subtopico:.3f}")

    return {
        "dupla": nome_dupla,
        "kappa_topico": kappa_topico,
        "kappa_subtopico": kappa_subtopico,
        "po_topico": po_topico,
        "po_subtopico": po_subtopico,
        "n": n
    }


def calcular_metricas_kappa_duplas(df):
    # Dupla 1: linhas 1 até 177 (índices 0 a 176)
    df_dupla_1 = df.iloc[:177]

    # Dupla 2: linha 178 até o final
    df_dupla_2 = df.iloc[177:]

    resultados = []
    resultados.append(calcular_kappa_segmento(df_dupla_1, "Dupla 1 (linhas 1–177)"))
    resultados.append(calcular_kappa_segmento(df_dupla_2, "Dupla 2 (linhas 178–final)"))

    return resultados


# Execução
calcular_metricas_kappa_duplas(df)



Resultados — Dupla 1 (linhas 1–177)
Total de registros: 177
----------------------------------------
TÓPICOS:
  - Porcentagem de Concordância: 97.18%
  - Valor de Kappa: 0.944

SUBTÓPICOS:
  - Porcentagem de Concordância: 94.35%
  - Valor de Kappa: 0.887

Resultados — Dupla 2 (linhas 178–final)
Total de registros: 178
----------------------------------------
TÓPICOS:
  - Porcentagem de Concordância: 98.88%
  - Valor de Kappa: 0.978

SUBTÓPICOS:
  - Porcentagem de Concordância: 96.07%
  - Valor de Kappa: 0.921


[{'dupla': 'Dupla 1 (linhas 1–177)',
  'kappa_topico': np.float64(0.9435028248587571),
  'kappa_subtopico': np.float64(0.887005649717514),
  'po_topico': np.float64(0.9717514124293786),
  'po_subtopico': np.float64(0.943502824858757),
  'n': 177},
 {'dupla': 'Dupla 2 (linhas 178–final)',
  'kappa_topico': np.float64(0.9775280898876404),
  'kappa_subtopico': np.float64(0.9213483146067416),
  'po_topico': np.float64(0.9887640449438202),
  'po_subtopico': np.float64(0.9606741573033708),
  'n': 178}]